In [1]:
import requests
import time

base_url = "https://collectionapi.metmuseum.org"
search_path = "/public/collection/v1.1/search"

search_query = "Native American"
limit = 500
offset = 0

all_results = []

while True:
    url = (
        f"{base_url}{search_path}"
        f"?q={search_query.replace(' ', '%20')}"
        f"&artistOrCulture=true"
        f"&limit={limit}"
        f"&offset={offset}"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    # Get the object IDs returned by this page
    object_ids = data.get("objectIDs") or []

    # Stop if there are no more results
    if not object_ids:
        break

    all_results.extend(object_ids)

    print(f"Collected {len(all_results)} records so far")

    # Move to the next page
    offset += limit

    time.sleep(1)

print(f"Total records collected: {len(all_results)}")

Collected 500 records so far
Collected 520 records so far
Total records collected: 520


In [2]:
print(url)

https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=Native%20American&artistOrCulture=true&limit=500&offset=1000


In [3]:
complete_nativeamerican_objects = all_results

print(len(complete_nativeamerican_objects))


520


In [5]:
from time import sleep 

object_path = "/public/collection/v1/objects/" 
all_nativeamerican_details = []

for nativeamerican_artwork in complete_nativeamerican_objects:
    url = f"{base_url}{object_path}{nativeamerican_artwork}"
    response_object = requests.get(url) 
    
    # Check if the API request was successful (status code 200)
    if response_object.ok:
        all_nativeamerican_details.append(response_object.json())
        print(f"grabbed details from objectID {nativeamerican_artwork} with status code {response_object.status_code}")
    else:
        # If it's a 404 or any other error, skip it and don't append it to the list
        print(f"⚠️ SKIPPED objectID {nativeamerican_artwork} - Received status code {response_object.status_code}")
        
    sleep(1) 

print(f"Total valid objects captured: {len(all_nativeamerican_details)}") 
if all_nativeamerican_details:
    print(all_nativeamerican_details[0])

grabbed details from objectID 120950 with status code 200
grabbed details from objectID 27229 with status code 200
grabbed details from objectID 500882 with status code 200
grabbed details from objectID 500888 with status code 200
grabbed details from objectID 500889 with status code 200
grabbed details from objectID 501178 with status code 200
grabbed details from objectID 501180 with status code 200
grabbed details from objectID 501189 with status code 200
grabbed details from objectID 501191 with status code 200
grabbed details from objectID 501192 with status code 200
grabbed details from objectID 501194 with status code 200
grabbed details from objectID 501197 with status code 200
grabbed details from objectID 501200 with status code 200
grabbed details from objectID 501203 with status code 200
grabbed details from objectID 501204 with status code 200
grabbed details from objectID 501205 with status code 200
grabbed details from objectID 501213 with status code 200
grabbed details

In [6]:
import csv   # brings in all the csv-related functions

with open("all_nativeamerican_artworks.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, all_nativeamerican_details[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(all_nativeamerican_details) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [7]:
nativeamerican_with_state = [
    artwork for artwork in all_nativeamerican_details
    if artwork.get("state")
]

In [8]:
print(f"Original records: {len(all_nativeamerican_details)}")
print(f"Records with state: {len(nativeamerican_with_state)}")

Original records: 501
Records with state: 361


In [9]:
import csv   # brings in all the csv-related functions

with open("nativeamerican_artworks_withstate.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, nativeamerican_with_state[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(nativeamerican_with_state) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [10]:
nativeamerican_with_state_filtered = [
    artwork for artwork in all_nativeamerican_details
    if artwork.get("state")
    and not any(term in artwork["state"] for term in ["and", "or", "/", ",", "?", "&"])
]

In [11]:
print(f"Original records: {len(all_nativeamerican_details)}")
print(f"Records with state: {len(nativeamerican_with_state)}")
print(f"Records with state: {len(nativeamerican_with_state_filtered)}")

Original records: 501
Records with state: 361
Records with state: 276


In [12]:
import csv   # brings in all the csv-related functions

with open("nativeamerican_artworks_withstate_filtered.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, nativeamerican_with_state[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(nativeamerican_with_state_filtered) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [15]:
import pandas as pd

# Read the bigger CSV for unique cultures to create colors
df= pd.read_csv("nativeamerican_artworks_with_coordinates.csv")
df['culture'].value_counts()

culture
Native American (Northwest Coast)                                         48
Xaadas (Haida), Native American                                           16
Skittagetan, probably Xaadas (Haida), Native American                      9
Tlingit, Native American                                                   7
Native American (Zuni)                                                     6
                                                                          ..
Makah or Nuu-chah-nulth, Native American                                   1
Native American (Salish or Nuu-chah-nulth - Nootka)                        1
Probably Xaadas (Haida) or T'smsyen (Tsimshian), Native American           1
Native American (Tsa-Tinne?, Northern Athapascan) or (Southern Plains)     1
T'smsyen (Tsimshian)?, Native Ameircan                                     1
Name: count, Length: 149, dtype: int64

In [16]:
pd.set_option('display.max_rows', None)

df['culture'].value_counts()

culture
Native American (Northwest Coast)                                         48
Xaadas (Haida), Native American                                           16
Skittagetan, probably Xaadas (Haida), Native American                      9
Tlingit, Native American                                                   7
Native American (Zuni)                                                     6
Native American (Hopi)                                                     5
Native American (Tlingit)                                                  5
Native American (Guyanese: Demerara)                                       5
Yup'ik, Native American                                                    3
Haida, Native American                                                     3
Native American (Guyanese - Demerara)                                      3
Native American (Nootka)                                                   3
Kiowa, Native American                                              

In [17]:
pd.set_option('display.max_rows', None)

df['state'].value_counts()

state
British Columbia     112
Alaska                45
Arizona               22
New Mexico            22
Demerara              12
Oklahoma               9
South Dakota           6
Washington             6
Ontario                5
Wyoming                4
Minnesota              3
Great Plains           3
Quebec                 3
Oregon                 2
Nevada                 2
Dakotas                2
Alabama                2
Matto Grosso           1
Nebraska               1
Valley of Mexico       1
Nova Scotia            1
New Brunswick          1
Maine                  1
Utah                   1
Montana                1
Amazonas province      1
Michigan               1
Louisiana              1
Puerto Rico            1
Ohio                   1
Pennsylvania           1
Wisconsin              1
Iowa                   1
Name: count, dtype: int64